# 07 · Metals

A metal is the same `embed`, given an arrangement. `rx.metal(source, geometry)` enumerates the distinct
coordination isomers; pick one and embed it.

The metal is swapped to a bond-less surrogate for the force field, because UFF cannot type a bonded
transition metal, and restored with its oxidation state and dative bonds on the way out.


In [ ]:
import tempfile
import numpy as np
import rxembed.pipeline as rx
import xyzrender
from rdkit import RDLogger
from rdkit.Chem import rdMolTransforms as T
from rxembed.pipeline.perceive import _xyz_to_mol
from rxembed.metal_core import metal_indices

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")


def xyz(ens):
    return ens.lowest(1).dump(tempfile.mktemp(suffix=".xyz"))  # 1 conformer -> a path xyzrender reads


def mxyz(ens):
    return ens.dump(tempfile.mktemp(suffix=".xyz"))

## Coordination isomers: the permutations

Square-planar MA₂B₂ → cis / trans; octahedral MA₃B₃ → mer / fac. `.summary()` tags each with its per-vertex arrangement and Λ/Δ chirality, the unambiguous, name-agnostic identity. Pick one with `.select(index=…)`, `.select(arrangement=…)`, or `.select(chirality=…)` (the coarse cis/trans/mer/fac name is intentionally not the selector, it's ambiguous once several isomers share it).

In [ ]:
print("square-planar Pd (cis/trans):")
rx.metal("CCCN[Pd](Cl)(Cl)NCCC", "square_planar").summary()
print("\noctahedral Co(NH3)3Cl3 (mer/fac):")
rx.metal("N[Co](N)(N)(Cl)(Cl)Cl", "octahedral").summary()

The fac Co isomer, its coordination sphere drawn as dotted dative bonds.


In [ ]:
co = rx.metal("[NH3][Co](N)(N)(Cl)(Cl)Cl", "octahedral").select(index=1)  # the fac isomer (index 1 in .summary())
ens = rx.embed(co).mc(preset="ensemble").prune()
p = xyz(ens)
xyzrender.render(p, no_hy=True, idx="s")

The trans Pd isomer: the two Cl sit about 180° apart across the metal.


In [ ]:
trans = rx.metal("CCCN[Pd](Cl)(Cl)NCCC", "square_planar").select(index=1)  # the trans isomer (index 1)
ens = rx.embed(trans).mc(preset="ensemble").prune()
cl = [d for d in trans.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "Cl"]
print(f"{len(ens)} conformers | Cl-Pd-Cl = {ens.measure((cl[0], trans.metal, cl[1]))['mean']:.0f}° (trans)")
p = xyz(ens)
xyzrender.render(
    p, no_hy=True, labels=[f"{cl[0] + 1} {trans.metal + 1} {cl[1] + 1} a"]
)  # label the trans Cl-Pd-Cl angle

## Across coordination geometries

One surface spans the polyhedra; linear, tetrahedral, square-planar, octahedral (and trigonal-bipyramidal / square-pyramidal for 5-coordinate).

In [ ]:
for g, smi in {
    "linear": "[Cl][Au][Cl]",
    "tetrahedral": "Cl[Zn](Cl)(Cl)Cl",
    "square_planar": "CCCN[Pd](Cl)(Cl)NCCC",
    "octahedral": "N[Co](N)(N)(Cl)(Cl)Cl",
}.items():
    isos = rx.metal(smi, g)
    print(f"  {g:14s}: {len(isos)} isomer(s) {[i.label for i in isos]}")

## Metal-centre chirality

Λ and Δ, from a point-group parity construction rather than RDKit's native metal stereo, whose permutation
is not order-invariant for equivalent ligands.


In [ ]:
isos = rx.metal("Cl[Co]12(Cl)(<-NCC<-N1)<-NCC<-N2", "octahedral")  # [Co(en)2Cl2]; bis-chelate + 2 Cl
isos.summary()  # trans | achiral ;  cis | Δ ;  cis | Λ
delta = isos.select(chirality="Δ")  # pick the Δ enantiomer by its handedness (name-agnostic, unambiguous)
ens = rx.embed(delta, n=8).minimize().mc(preset="ensemble").minimize().prune()  # relax -> search -> re-relax -> dedup
reps = ens.representatives()
print(f"cis-Δ [Co(en)2Cl2]: {ens.n} conformers -> {reps.n} distinct mode(s)")
rep = rx.geom_check.check(reps.mol, conf_id=reps.ids[0], donors=list(delta.donors))
print("geometry.check:", "CLEAN ✓" if rep.ok() else f"FAIL ✗ {rep.violations[:4]}")  # incl. folded donors
xyzrender.render(reps[0].dump(tempfile.mktemp(suffix=".xyz")), no_hy=True, idx="s")  # reps[0]: the new [i] indexing

## Chelating ligands: bidentate

A chelate's donors sit on one ligand fragment; rxembed detects same-fragment donors and widens their L–M–L window (a rigid backbone bite is distorted off the ideal) while still separating cis from trans. Ethylenediamine on Pd (a 5-membered chelate ring, N–Pd–N bite ~82°) + two chloride:

In [ ]:
en = rx.metal("Br[Pd]1(Cl)<-NCC<-N1", "square_planar")  # the NCCN ring pins both N to Pd
iso = en.select()
ens = rx.embed(iso, n=10).minimize().mc(preset="ensemble").minimize().prune()
nn = [d for d in iso.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "N"]
print(f"{len(ens)} conformers | en bite N-Pd-N = {ens.measure((nn[0], iso.metal, nn[1]))['mean']:.0f}°")
xyzrender.render(
    xyzrender.load(
        ens.dump(tempfile.mktemp(suffix=".xyz")), ensemble=len(ens) > 1, ensemble_color="spectral", nci_detect=True
    ),
    no_hy=True,
)

## Side-on η² coordination

When a ligand's two coordinating atoms are directly π-bonded (an alkene, alkyne, imine, phosphaalkene), it binds *side-on*: both atoms grip through the π bond. rxembed detects the multiple-bonded donor pair and holds that bond at its natural length, so the two metal pulls can't stretch the C=C / C≡C apart (which would otherwise tear it and blow the coordination sphere out). Here a Ni acyl-malonate with a side-on C=C: both Ni–C distances bind and the double bond stays intact.

In [ ]:
iso = rx.metal(
    "COC(=O)[C]12->[Ni+2]3(<-[O-]C(=O)C(c4ccccc4)[N-]->3c3ccccc3)<-[C]=1(C(=O)OC)C2(C)C(C)(C)C",
    "square_planar",
).select(index=0)
pi = next(  # the directly π-bonded donor pair: the side-on C=C
    (a, b)
    for a in iso.donors
    for b in iso.donors
    if a < b and (bd := iso.mol.GetBondBetweenAtoms(a, b)) and bd.GetBondTypeAsDouble() >= 2
)
ens = rx.embed(iso, n=8).minimize().mc(preset="ensemble").minimize().prune()
c = ens.mol.GetConformer(ens.ids[0])
print(
    f"side-on C=C {pi}: {ens.n} confs | Ni-C {T.GetBondLength(c, iso.metal, pi[0]):.2f}/"
    f"{T.GetBondLength(c, iso.metal, pi[1]):.2f} Å (both bind) | C=C {T.GetBondLength(c, *pi):.2f} Å (held intact)"
)
xyzrender.render(xyz(ens), no_hy=True, labels=[f"{iso.metal + 1} {pi[0] + 1} d", f"{iso.metal + 1} {pi[1] + 1} d"])

## Vacant sites: a coordination pocket

Name a geometry with more vertices than donors and the empty vertex is left as a pocket (`·` in the arrangement) for a substrate to bind later. A 3-donor Pd in a square-planar field leaves one site open.

In [ ]:
pocket = rx.metal("CC#N->[Pd](Cl)Cl", "square_planar")  # N + 2 Cl = 3 donors, 4 vertices
pocket.summary()  # '·' marks the vacant coordination site
iso = pocket.select(index=0)  # the cis pocket (both Cl cis; the '·' vacancy sits opposite the lone Cl)
ens = rx.embed(iso).mc(preset="ensemble").prune()
xyzrender.render(xyz(ens), no_hy=True, idx="s")  # the '·' vacancy sits opposite the lone Cl